In [1]:
import xarray as xr
import sys, os
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
from cloudpathlib import AnyPath

# Local Imports
curdir = os.getcwd()
#print(curdir)
sys.path.insert(0, curdir+"/../../../data_processing")
from ceres_ebaf import *

sys.path.insert(0, curdir+"/../../../visualization")
from ceres_ebaf_plotting import *

base_path = AnyPath("/Users/mawa7160/dev/data/CERES/")
full_ebaf_filename = "EBAF/CERES_EBAF-TOA_Ed4.2_Subset_200003-202402.nc"
syn_part_1 = "SYN/CERES_SYN1deg-Day_Terra-Aqua-MODIS_Ed4.1_Subset_20000301-20200325.nc"
syn_part_2 = "SYN/CERES_SYN1deg-Day_Terra-Aqua-MODIS_Ed4.1_Subset_20200326-20240229.nc"

plt.rcParams['figure.figsize'] = [10, 3.5]

In [2]:
# Load the full SYN1deg data
syn_part_1_data = xr.open_dataset(base_path / syn_part_1)
syn_part_2_data = xr.open_dataset(base_path / syn_part_2)   
syn = xr.concat([syn_part_1_data, syn_part_2_data], dim='time')

In [3]:
syn_global = apply_spatial_weights(syn["toa_solar_all_daily"], low_lat=-90, high_lat=90)
syn_nh = apply_spatial_weights(syn["toa_solar_all_daily"], low_lat=0, high_lat=90)
syn_sh = apply_spatial_weights(syn["toa_solar_all_daily"], low_lat=-90, high_lat=0)

In [4]:
syn_2019_global = syn_global.sel(time=slice("2018-12-31", "2020-01-01"))
syn_2019_nh = syn_nh.sel(time=slice("2018-12-31", "2020-01-01"))
syn_2019_sh = syn_sh.sel(time=slice("2018-12-31", "2020-01-01"))

plt.plot(syn_2019_global.time, syn_2019_global, "--xk", label="Global")
plt.plot(syn_2019_nh.time, syn_2019_nh, "--xg", label="NH")
plt.plot(syn_2019_sh.time, syn_2019_sh, "--xb", label="SH")
plt.legend()

In [5]:
numeric_offset = .12125-0.12018675
month_offsets = -numeric_offset*np.cos((np.arange(12)-10/31)*2*np.pi/12)

LEAP_YEAR_OFFSET = - (1-0.2425)/2 + month_offsets[0]
NON_LEAP_YEAR_OFFSET = 0.2425/2 + month_offsets[0]
def make_weighted_syn_annual_mean(syn_data, year, offset=0.2425/2):
    syn_year = syn_data.sel(time=slice(f"{year-1}-12-31", f"{year+1}-01-01"))
    weights = np.ones(len(syn_year))
    weights[0] = offset
    weights[-1] = offset
    syn_weighted = syn_year.weighted(xr.DataArray(weights, dims='time'))
    return syn_weighted.mean().values

In [6]:
print("2020 Global Mean: ", make_weighted_syn_annual_mean(syn_global, 2019))
print("NH Mean: ", make_weighted_syn_annual_mean(syn_nh, 2019))
print("SH Mean: ", make_weighted_syn_annual_mean(syn_sh, 2019))

In [7]:
years = np.arange(21)+2003
global_means = np.ones(len(years), dtype=float)
nh_means = np.ones_like(global_means)
sh_means = np.ones_like(global_means)
for i in range(len(years)):
    if years[i] % 4 == 0:
        offset = LEAP_YEAR_OFFSET
    else:
        offset = NON_LEAP_YEAR_OFFSET
    global_means[i] = make_weighted_syn_annual_mean(syn_global, years[i], offset=offset)
    nh_means[i] = make_weighted_syn_annual_mean(syn_nh, years[i], offset=offset)
    sh_means[i] = make_weighted_syn_annual_mean(syn_sh, years[i], offset=offset)
    

In [8]:
plt.plot(years, global_means)
plt.plot(years, nh_means)
plt.plot(years, sh_means)

In [26]:
years = np.arange(21)+2003

predictions = np.zeros((4, 6), dtype=float)
years_array = np.zeros((4, 6), dtype=float)
estimates = np.zeros((4, 6), dtype=float)
for i in range(len(years)):
    leap_index = years[i] % 4
    step_index = np.floor((years[i] - 2003)/4).astype(int)
    years_array[leap_index, step_index] = years[i]
    predictions[leap_index, step_index] = global_means[i]

print(years_array)

In [10]:
from scipy.optimize import minimize

def optimize_h_vs_global(ref_data, year_data, prediction_data, initial_guess):
    
    def model(years, offset_early, offset_late):
        results = np.array([])
        for year in years:
            year_int = year.astype(int)
            syn_year = ref_data.sel(time=slice(f"{year_int-1}-12-31", f"{year_int+1}-01-01"))
            weightings = np.ones(len(syn_year))
            weightings.put(0, offset_early)
            weightings.put(-1, offset_late)
            syn_weighted = syn_year.weighted(xr.DataArray(weightings, dims='time'))
            mean = syn_weighted.mean().values
            results = np.append(results, mean)
        return results
    
    def objective(params):
        offset_early, offset_late = params
        y_est = model(year_data, offset_early, offset_late)
        error = prediction_data - y_est
        return np.sum(error**2)
    
    result = minimize(objective, initial_guess)
    return result.x

In [11]:
optimizations_nh = np.empty((4,2))
optimizations_sh = np.empty((4,2))
for i in range(4):
    if i !=3:
        years_local = years_array[i, :-1]
        predictions_local = predictions[i, :-1]
    else:
        years_local = years_array[i,:]
        predictions_local = predictions[i,:]
        
    if i==0:
        local_initial_guess = 0.12125
    else:
        local_initial_guess = (1-0.2425)/2
    print(years_local)
    optimizations_nh[i] = optimize_h_vs_global(syn_nh, years_local, predictions_local, [local_initial_guess, local_initial_guess])
    optimizations_sh[i] =  optimize_h_vs_global(syn_sh, years_local, predictions_local, [local_initial_guess, local_initial_guess])
print(optimizations_nh)
print(optimizations_sh)

In [12]:
original_guess = np.ones_like(optimizations_nh)*0.12125
original_guess.put((0,0), -(1-0.2425)/2)
original_guess.put((0,1), -(1-0.2425)/2)

In [13]:
print(original_guess-optimizations_nh)

In [14]:
plt.plot((original_guess-optimizations_nh)[:,0])
plt.plot((original_guess-optimizations_sh)[:,0])

In [15]:
# Calculate the distance to the sun using astropy
from astropy.time import Time
from astropy.coordinates import get_sun
from astropy.constants import au
from datetime import date

def earth_to_sun_distance(date):
    # Parse the given date
    t = Time(date)

    # Get the Sun's position from the Earth
    sun = get_sun(t)

    # Calculate the distance in astronomical units
    distance_au = sun.distance / au

    return distance_au.value

In [19]:
dates = [f"{year}-12-15" for year in [2003, 2004, 2005, 2006]]
distances = [earth_to_sun_distance(date) for date in dates]

In [20]:
distances

In [21]:
plt.plot(distances)

In [31]:
avg_distance_before = np.empty(4)
for i in range(4):
    if i !=3:
        local_years = years_array[i, :-1]
    else:
        local_years = years_array[i, :]
    print(local_years)
    dates = [f"{int(year-1)}-12-15" for year in local_years]
    distances = [earth_to_sun_distance(date) for date in dates]
    avg_distance_before[i] = np.mean(distances)

In [32]:
plt.plot(avg_distance_before)

In [33]:
def monthly_avg_distance(year, month):
    dates = [f"{int(year)}-{month:02}-{day+1:02}" for day in range(31)]
    distances = [earth_to_sun_distance(date) for date in dates]
    return np.mean(distances)

In [34]:
month = 12
o4_s = monthly_avg_distance(2003, month)
o5_s = monthly_avg_distance(2004, month)
o6_s = monthly_avg_distance(2005, month)
o7_s = monthly_avg_distance(2006, month)

In [35]:
plt.plot([o4_s, o5_s, o6_s, o7_s])

In [36]:
month = 1
o4_e = monthly_avg_distance(2005, month)
o5_e = monthly_avg_distance(2006, month)
o6_e = monthly_avg_distance(2007, month)
o7_e = monthly_avg_distance(2008, month)

In [37]:
plt.plot([o4_e, o5_e, o6_e, o7_e])

In [38]:
o4 = (o4_s + o4_e)/2
o5 = (o5_s + o5_e)/2
o6 = (o6_s + o6_e)/2
o7 = (o7_s + o7_e)/2

plt.plot([o4, o5, o6, o7])

In [46]:
optimizations = np.concatenate((optimizations_nh, optimizations_sh), axis=1)

def make_accurate_weighted_syn_annual_mean(syn_data, year, hemi):
    syn_year = syn_data.sel(time=slice(f"{year-1}-12-31", f"{year+1}-01-01"))
    weights = np.ones(len(syn_year))
    if hemi == "NH":
        hemi_offset = 0
    elif hemi == "SH":
        hemi_offset = 2
    weights[0] = optimizations[year%4, hemi_offset]
    weights[-1] = optimizations[year%4, hemi_offset+1]
    syn_weighted = syn_year.weighted(xr.DataArray(weights, dims='time'))
    return syn_weighted.mean().values

print(optimizations)

In [57]:
years = np.arange(20)+2003
global_means = np.ones(len(years), dtype=float)
nh_means = np.ones_like(global_means)
sh_means = np.ones_like(global_means)
for i in range(len(years)):
    global_means[i] = make_accurate_weighted_syn_annual_mean(syn_global, years[i], "SH")
    nh_means[i] = make_accurate_weighted_syn_annual_mean(syn_nh, years[i], "NH")
    sh_means[i] = make_accurate_weighted_syn_annual_mean(syn_sh, years[i], "SH")

In [58]:
plt.plot(years, global_means)
plt.plot(years, nh_means)
plt.plot(years, sh_means)